# 09 - Embedding Space 分析（t-SNE / UMAP）

**目標**：視覺化 CNN / LSTM / wav2vec 三種模型的中間層 embedding，
驗證「pre-trained representation 的 emotion cluster 分離度優於傳統特徵」。

- **輸入**：`data/embeddings/{cnn,lstm,wav2vec}_embeddings.npy` + `embedding_meta.csv`
- **產出**：Fig 12 (t-SNE by emotion)、Fig 13 (t-SNE by dataset)、Fig 14 (UMAP by emotion)、Silhouette score

> 此 notebook 在本地端執行（CPU 即可）。

In [ ]:
# === Setup ===
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

try:
    import umap
    HAS_UMAP = True
except ImportError:
    print('umap-learn not installed. Run: pip install umap-learn')
    HAS_UMAP = False

PROJECT_ROOT = Path('..').resolve()
EMB_DIR = PROJECT_ROOT / 'data' / 'embeddings'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

# === 載入 embedding ===
meta = pd.read_csv(EMB_DIR / 'embedding_meta.csv')
embeddings = {}
for name in ['cnn', 'lstm', 'wav2vec']:
    path = EMB_DIR / f'{name}_embeddings.npy'
    if path.exists():
        embeddings[name] = np.load(path)
        print(f'{name}: shape={embeddings[name].shape}')
    else:
        print(f'[MISSING] {path.name} — 請先執行 embedding 萃取')

print(f'\nMeta: {len(meta)} rows')
print(f'Emotions: {sorted(meta["emotion"].unique())}')
print(f'Datasets: {sorted(meta["dataset"].unique())}')

In [ ]:
# === t-SNE 計算（快取結果避免重複計算） ===
# 每個模型的 t-SNE 約 2-5 分鐘

tsne_results = {}

for name, emb in embeddings.items():
    cache_path = EMB_DIR / f'{name}_tsne2d.npy'
    if cache_path.exists():
        tsne_results[name] = np.load(cache_path)
        print(f'{name}: loaded cached t-SNE — shape {tsne_results[name].shape}')
    else:
        print(f'{name}: computing t-SNE (perplexity=30, n_iter=1000)...')
        tsne = TSNE(n_components=2, perplexity=30, n_iter=1000,
                    random_state=RANDOM_STATE, init='pca', learning_rate='auto')
        result = tsne.fit_transform(emb)
        np.save(cache_path, result)
        tsne_results[name] = result
        print(f'{name}: done — shape {result.shape}, saved to {cache_path.name}')

In [ ]:
# === UMAP 計算（快取結果） ===

umap_results = {}

if HAS_UMAP:
    for name, emb in embeddings.items():
        cache_path = EMB_DIR / f'{name}_umap2d.npy'
        if cache_path.exists():
            umap_results[name] = np.load(cache_path)
            print(f'{name}: loaded cached UMAP — shape {umap_results[name].shape}')
        else:
            print(f'{name}: computing UMAP (n_neighbors=15, min_dist=0.1)...')
            reducer = umap.UMAP(n_neighbors=15, min_dist=0.1,
                                n_components=2, random_state=RANDOM_STATE)
            result = reducer.fit_transform(emb)
            np.save(cache_path, result)
            umap_results[name] = result
            print(f'{name}: done — shape {result.shape}')
else:
    print('UMAP skipped (umap-learn not installed)')

In [ ]:
# === Fig 12: t-SNE by Emotion (3 panels) ===

EMOTION_COLORS = {
    'angry': '#e74c3c',
    'disgust': '#8e44ad',
    'fear': '#f39c12',
    'happy': '#2ecc71',
    'neutral': '#3498db',
    'sad': '#1abc9c',
}

model_names = [m for m in ['cnn', 'lstm', 'wav2vec'] if m in tsne_results]
fig = make_subplots(
    rows=1, cols=len(model_names),
    subplot_titles=[m.upper() for m in model_names],
    horizontal_spacing=0.05,
)

emotions = sorted(meta['emotion'].unique())

for col_idx, model_name in enumerate(model_names, 1):
    pts = tsne_results[model_name]
    for emo in emotions:
        mask = meta['emotion'] == emo
        fig.add_trace(
            go.Scattergl(
                x=pts[mask, 0], y=pts[mask, 1],
                mode='markers',
                marker=dict(size=2, color=EMOTION_COLORS[emo], opacity=0.5),
                name=emo,
                legendgroup=emo,
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )

fig.update_layout(
    title='t-SNE Embedding Space — Colored by Emotion',
    height=500, width=1400, template='plotly_white',
    legend=dict(title='Emotion', itemsizing='constant'),
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

fig.write_html(FIG_DIR / '12_tsne_by_emotion.html')
fig.write_image(FIG_DIR / '12_tsne_by_emotion.png', width=1400, height=500, scale=2)
fig.show()

In [ ]:
# === Fig 13: t-SNE by Dataset (3 panels) ===

DATASET_COLORS = {
    'RAVDESS': '#e74c3c',
    'CREMA-D': '#3498db',
    'TESS': '#2ecc71',
    'SAVEE': '#f39c12',
}

fig = make_subplots(
    rows=1, cols=len(model_names),
    subplot_titles=[m.upper() for m in model_names],
    horizontal_spacing=0.05,
)

datasets = sorted(meta['dataset'].unique())

for col_idx, model_name in enumerate(model_names, 1):
    pts = tsne_results[model_name]
    for ds in datasets:
        mask = meta['dataset'] == ds
        fig.add_trace(
            go.Scattergl(
                x=pts[mask, 0], y=pts[mask, 1],
                mode='markers',
                marker=dict(size=2, color=DATASET_COLORS[ds], opacity=0.5),
                name=ds,
                legendgroup=ds,
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )

fig.update_layout(
    title='t-SNE Embedding Space — Colored by Dataset',
    height=500, width=1400, template='plotly_white',
    legend=dict(title='Dataset', itemsizing='constant'),
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

fig.write_html(FIG_DIR / '13_tsne_by_dataset.html')
fig.write_image(FIG_DIR / '13_tsne_by_dataset.png', width=1400, height=500, scale=2)
fig.show()

In [ ]:
# === Fig 14: UMAP by Emotion (3 panels) ===

if umap_results:
    fig = make_subplots(
        rows=1, cols=len(model_names),
        subplot_titles=[m.upper() for m in model_names],
        horizontal_spacing=0.05,
    )

    for col_idx, model_name in enumerate(model_names, 1):
        pts = umap_results[model_name]
        for emo in emotions:
            mask = meta['emotion'] == emo
            fig.add_trace(
                go.Scattergl(
                    x=pts[mask, 0], y=pts[mask, 1],
                    mode='markers',
                    marker=dict(size=2, color=EMOTION_COLORS[emo], opacity=0.5),
                    name=emo,
                    legendgroup=emo,
                    showlegend=(col_idx == 1),
                ),
                row=1, col=col_idx,
            )

    fig.update_layout(
        title='UMAP Embedding Space — Colored by Emotion',
        height=500, width=1400, template='plotly_white',
        legend=dict(title='Emotion', itemsizing='constant'),
    )
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)

    fig.write_html(FIG_DIR / '14_umap_by_emotion.html')
    fig.write_image(FIG_DIR / '14_umap_by_emotion.png', width=1400, height=500, scale=2)
    fig.show()
else:
    print('UMAP results not available — skipped')

In [ ]:
# === Silhouette Score（量化 cluster 分離度） ===
# 用 emotion label 計算 silhouette score，數值越高代表 cluster 越好

import json

sil_scores = {}
labels = meta['emotion'].values

for name, emb in embeddings.items():
    score = silhouette_score(emb, labels, metric='euclidean', sample_size=5000,
                             random_state=RANDOM_STATE)
    sil_scores[name] = round(score, 4)
    print(f'{name:>8s} silhouette score: {score:.4f}')

# 儲存供 NB10 使用
sil_path = EMB_DIR / 'silhouette_scores.json'
with open(sil_path, 'w') as f:
    json.dump(sil_scores, f, indent=2)
print(f'\nSaved: {sil_path}')

## 討論

### 預期觀察

1. **t-SNE by emotion (Fig 12)**：
   - wav2vec 的 6 個 emotion cluster 應該明顯分離
   - CNN 有部分分離但交疊較多
   - LSTM 的 cluster 最模糊

2. **t-SNE by dataset (Fig 13)**：
   - CNN/LSTM 的點可能按 dataset 分群（domain-specific feature）
   - wav2vec 應該更 dataset-agnostic（不同 dataset 的同情緒點混在一起）

3. **Silhouette score**：
   - 預期排序：wav2vec > CNN > LSTM
   - 與 in-corpus accuracy 呈正相關
   - 量化支持「feature representation quality → classification performance」

### Limitations

#### 1. Fold-1 Embedding 的方法論限制

Embedding 視覺化採用 fold-1 best checkpoint 對**全資料**（11,318 筆）萃取，其中約 80% 的樣本為該 fold 的 training data。這意味著模型對這些樣本可能存在過擬合，使得 t-SNE 上的 cluster 看起來比實際更「乾淨」，silhouette score 也可能被膨脹。

然而，由於三個模型使用**相同的 protocol**（皆為 fold-1 萃取全資料），模型之間的**相對差異仍然有效**。本分析的目的是呈現三種特徵表徵的相對品質差異，而非報告絕對的 cluster 品質指標。

更嚴謹的做法是採用 **out-of-fold prediction**：用 5 個 fold 各自的 test set 拼接，使每個樣本的 embedding 都來自模型「未見過」的狀態。此方法留待未來工作。

#### 2. 高維空間 Silhouette Score 的限制

實際觀測到 CNN silhouette (-0.0008) < LSTM (0.0191)，但 CNN 的 in-corpus accuracy (58.6%) 明顯高於 LSTM (49.4%)。這一看似矛盾的現象有合理解釋：

- Silhouette score 基於 **euclidean distance**，在 256 維空間中容易受到維度災難（curse of dimensionality）影響
- CNN 的 block4 特徵可能形成**非球狀但線性可分**的 cluster，這種結構對 classifier 有效，但不會被基於距離的 silhouette 指標捕捉
- wav2vec 的 silhouette (0.3303) 遠高於 CNN/LSTM，說明其 768 維 embedding 不僅可分，而且在 euclidean 空間中就已形成良好的球狀 cluster

因此，silhouette score 應作為**輔助性**的 cluster 品質指標，而非唯一的特徵品質判據。結合 t-SNE 視覺化與分類準確率一起解讀更為完整。